# Merge BBC URNs into player_apps.csv

This notebook reads Tranmere lineup data from `bbc-json/lineups`, flattens the BBC player records, and merges BBC player URNs into `../prepare-sql-db/csv-prepared/player_apps.csv`.

Matching assumptions:
- Only the Tranmere side of each BBC lineup is used, even when Tranmere is the away team.
- BBC `starters` are mapped to CSV role `starter`.
- BBC `substitutes` are mapped to CSV role `sub`.
- The merge key is `game_date + role + shirt_no`.
- Rows outside the BBC lineup date range remain unmatched.

In [5]:
from pathlib import Path
import json

import pandas as pd

WORKSPACE_ROOT = Path.cwd()
BBC_LINEUPS_DIR = WORKSPACE_ROOT / "bbc-json" / "lineups"
PLAYER_APPS_PATH = (WORKSPACE_ROOT.parent / "prepare-sql-db" / "csv-prepared" / "player_apps.csv").resolve()
OUTPUT_PATH = WORKSPACE_ROOT / "data" / "player_apps_with_bbc_urn.csv"
TARGET_TEAM_URN = "urn:bbc:sportsdata:football:team:tranmere-rovers"
TARGET_TEAM_CODE = "TRA"

if not BBC_LINEUPS_DIR.exists():
    raise FileNotFoundError(BBC_LINEUPS_DIR)

if not PLAYER_APPS_PATH.exists():
    raise FileNotFoundError(PLAYER_APPS_PATH)

print(f"BBC lineups: {BBC_LINEUPS_DIR}")
print(f"player_apps.csv: {PLAYER_APPS_PATH}")
print(f"Output file: {OUTPUT_PATH}")

BBC lineups: /Users/petebrown/Developer/data-updater-v2/bbc-json/lineups
player_apps.csv: /Users/petebrown/Developer/prepare-sql-db/csv-prepared/player_apps.csv
Output file: /Users/petebrown/Developer/data-updater-v2/data/player_apps_with_bbc_urn.csv


In [6]:
def find_target_team(payload):
    teams = [payload.get("homeTeam"), payload.get("awayTeam")]
    for team in teams:
        if not team:
            continue
        if team.get("urn") == TARGET_TEAM_URN:
            return team
        if team.get("name", {}).get("code") == TARGET_TEAM_CODE:
            return team
    return None


def normalize_role(role_key):
    role_map = {"starters": "starter", "substitutes": "sub"}
    return role_map[role_key]


def extract_bbc_players(lineups_dir):
    records = []
    missing_team_dates = []

    for json_path in sorted(lineups_dir.glob("*.json")):
        payload = json.loads(json_path.read_text())
        team = find_target_team(payload)

        if team is None:
            missing_team_dates.append(json_path.stem)
            continue

        players = team.get("players", {})
        team_name = team.get("name", {}).get("fullName")
        team_alignment = team.get("alignment")

        for role_key in ("starters", "substitutes"):
            for player in players.get(role_key, []):
                name = player.get("name", {})
                records.append(
                    {
                        "game_date": json_path.stem,
                        "role": normalize_role(role_key),
                        "shirt_no": player.get("shirtNumber"),
                        "bbc_urn": player.get("urn"),
                        "bbc_short_name": name.get("short"),
                        "bbc_first_name": name.get("first"),
                        "bbc_last_name": name.get("last"),
                        "bbc_display_name": player.get("displayName"),
                        "bbc_position": player.get("position"),
                        "bbc_team_name": team_name,
                        "bbc_team_alignment": team_alignment,
                        "bbc_source_file": json_path.name,
                    }
                )

    return pd.DataFrame.from_records(records), missing_team_dates

In [7]:
bbc_players, missing_team_dates = extract_bbc_players(BBC_LINEUPS_DIR)
join_keys = ["game_date", "role", "shirt_no"]
bbc_players["game_date"] = bbc_players["game_date"].astype("string")
bbc_players["role"] = bbc_players["role"].astype("string")
bbc_players["shirt_no"] = pd.to_numeric(bbc_players["shirt_no"], errors="coerce").astype("Int64")
bbc_key_duplicates = bbc_players[bbc_players.duplicated(join_keys, keep=False)].sort_values(join_keys)

if missing_team_dates:
    display(pd.Series(missing_team_dates, name="missing_team_date").to_frame())
    raise ValueError("Some BBC lineup files did not contain the target team.")

if not bbc_key_duplicates.empty:
    display(bbc_key_duplicates[join_keys + ["bbc_display_name", "bbc_urn", "bbc_source_file"]])
    raise ValueError("BBC lineup rows are not unique on game_date + role + shirt_no.")

display(bbc_players.head())
print(f"Extracted {len(bbc_players):,} Tranmere player rows from {bbc_players['game_date'].nunique():,} BBC lineup files.")

,game_date,role,shirt_no,bbc_urn,bbc_short_name,bbc_first_name,bbc_last_name,bbc_display_name,bbc_position,bbc_team_name,bbc_team_alignment,bbc_source_file
0,2020-01-01,starter,31,urn:bbc:sportsdata:football:player:s-c9vek5ubp...,A. Chapman,Aaron James,Chapman,A. Chapman,Goalkeeper,Tranmere Rovers,home,2020-01-01.json
1,2020-01-01,starter,2,urn:bbc:sportsdata:football:player:s-1a8msufvk...,C. Woods,Calum Jack,Woods,C. Woods,Defender,Tranmere Rovers,home,2020-01-01.json
2,2020-01-01,starter,5,urn:bbc:sportsdata:football:player:s-eabj99pro...,G. Ray,George Edward,Ray,G. Ray,Defender,Tranmere Rovers,home,2020-01-01.json
3,2020-01-01,starter,4,urn:bbc:sportsdata:football:player:s-2uwn00y18...,S. Nelson,Sidney Raymond Kenneth,Nelson,S. Nelson,Defender,Tranmere Rovers,home,2020-01-01.json
4,2020-01-01,starter,7,urn:bbc:sportsdata:football:player:s-3gffyf2cg...,K. Morris,Kieron,Morris,K. Morris,Midfielder,Tranmere Rovers,home,2020-01-01.json


Extracted 5,946 Tranmere player rows from 331 BBC lineup files.


In [8]:
player_apps = pd.read_csv(
    PLAYER_APPS_PATH,
    dtype={"game_date": "string", "player_id": "string", "role": "string"},
)
player_apps["game_date"] = player_apps["game_date"].astype("string")
player_apps["role"] = player_apps["role"].astype("string").str.strip().str.lower()
player_apps["shirt_no"] = pd.to_numeric(player_apps["shirt_no"], errors="coerce").astype("Int64")

bbc_dates = set(bbc_players["game_date"].dropna())
player_apps_on_bbc_dates = player_apps[player_apps["game_date"].isin(bbc_dates)].copy()

merged_player_apps = player_apps.merge(
    bbc_players[
        join_keys
        + [
            "bbc_urn",
            "bbc_short_name",
            "bbc_first_name",
            "bbc_last_name",
            "bbc_display_name",
            "bbc_position",
            "bbc_team_name",
            "bbc_team_alignment",
            "bbc_source_file",
        ]
    ],
    on=join_keys,
    how="left",
    validate="many_to_one",
)
merged_player_apps["matched_bbc_urn"] = merged_player_apps["bbc_urn"].notna()

summary = pd.DataFrame(
    {
        "rows": [
            len(player_apps),
            len(player_apps_on_bbc_dates),
            len(bbc_players),
            int(merged_player_apps["matched_bbc_urn"].sum()),
            int((~merged_player_apps["matched_bbc_urn"]).sum()),
            int(merged_player_apps.loc[merged_player_apps["game_date"].isin(bbc_dates), "matched_bbc_urn"].sum()),
            int((~merged_player_apps.loc[merged_player_apps["game_date"].isin(bbc_dates), "matched_bbc_urn"]).sum()),
        ]
    },
    index=[
        "player_apps rows (all dates)",
        "player_apps rows on BBC dates",
        "BBC lineup rows",
        "matched rows (all dates)",
        "unmatched rows (all dates)",
        "matched rows on BBC dates",
        "unmatched rows on BBC dates",
    ],
)

display(summary)
display(merged_player_apps.head())

,rows
player_apps rows (all dates),60092
player_apps rows on BBC dates,4740
BBC lineup rows,5946
matched rows (all dates),4740
unmatched rows (all dates),55352
matched rows on BBC dates,4740
unmatched rows on BBC dates,0


,game_date,player_id,role,shirt_no,bbc_urn,bbc_short_name,bbc_first_name,bbc_last_name,bbc_display_name,bbc_position,bbc_team_name,bbc_team_alignment,bbc_source_file,matched_bbc_urn
0,1921-08-27,BradshawHarry18950122,starter,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1921-08-27,GraingerJohn1896,starter,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1921-08-27,StuartTom18931025,starter,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1921-08-27,CampbellJohnny18941014,starter,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1921-08-27,MilnesCharles21885,starter,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


In [9]:
unmatched_on_bbc_dates = merged_player_apps[
    merged_player_apps["game_date"].isin(bbc_dates) & merged_player_apps["bbc_urn"].isna()
][ ["game_date", "player_id", "role", "shirt_no"] ].sort_values(["game_date", "role", "shirt_no"])

if unmatched_on_bbc_dates.empty:
    print("Every player_apps row on BBC-covered dates matched a BBC lineup row.")
else:
    display(unmatched_on_bbc_dates.head(50))
    print(f"Showing the first {min(len(unmatched_on_bbc_dates), 50):,} unmatched rows on BBC-covered dates.")

Every player_apps row on BBC-covered dates matched a BBC lineup row.


In [ ]:
# merged_player_apps.to_csv(OUTPUT_PATH, index=False)
# print(f"Wrote {len(merged_player_apps):,} rows to {OUTPUT_PATH}")
# display(merged_player_apps.loc[merged_player_apps["matched_bbc_urn"]].head())

Wrote 60,092 rows to /Users/petebrown/Developer/data-updater-v2/data/player_apps_with_bbc_urn.csv


,game_date,player_id,role,shirt_no,bbc_urn,bbc_short_name,bbc_first_name,bbc_last_name,bbc_display_name,bbc_position,bbc_team_name,bbc_team_alignment,bbc_source_file,matched_bbc_urn
55352,2020-01-01,WoodsCalum19870205,starter,2,urn:bbc:sportsdata:football:player:s-1a8msufvk...,C. Woods,Calum Jack,Woods,C. Woods,Defender,Tranmere Rovers,home,2020-01-01.json,True
55353,2020-01-01,NelsonSid19960101,starter,4,urn:bbc:sportsdata:football:player:s-2uwn00y18...,S. Nelson,Sidney Raymond Kenneth,Nelson,S. Nelson,Defender,Tranmere Rovers,home,2020-01-01.json,True
55354,2020-01-01,RayGeorge19931013,starter,5,urn:bbc:sportsdata:football:player:s-eabj99pro...,G. Ray,George Edward,Ray,G. Ray,Defender,Tranmere Rovers,home,2020-01-01.json,True
55355,2020-01-01,MorrisKieron19940603,starter,7,urn:bbc:sportsdata:football:player:s-3gffyf2cg...,K. Morris,Kieron,Morris,K. Morris,Midfielder,Tranmere Rovers,home,2020-01-01.json,True
55356,2020-01-01,FerrierMorgan19941115,starter,10,urn:bbc:sportsdata:football:player:s-1uer7zbsj...,M. Ferrier,Morgan James,Ferrier,M. Ferrier,Striker,Tranmere Rovers,home,2020-01-01.json,True


In [23]:
ids_and_bbc_urns = merged_player_apps[['player_id', 'bbc_urn']].copy().drop_duplicates().dropna().sort_values('player_id')

ids_and_bbc_urns.to_csv(WORKSPACE_ROOT / "data" / "external_ids.csv", index=False)
print(f"Wrote {len(ids_and_bbc_urns):,} player_id + bbc_urn rows to {WORKSPACE_ROOT / 'data' / 'external_ids.csv'}")

Wrote 133 player_id + bbc_urn rows to /Users/petebrown/Developer/data-updater-v2/data/external_ids.csv
